# Day 27: Client Code — Streaming, Async & Protocol Support
> *Inference Engineering* — Chapter 7.5 | Philip Kiely, Baseten Books 2026

**Layer:** Tooling | **Prerequisite:** Day 26 (Observability)


## Concept Overview

Production inference clients require async I/O, streaming response handling, and robust error recovery. Streaming uses Server-Sent Events (SSE) over HTTP — the server sends token chunks as they're generated, reducing perceived latency for users. Async clients (asyncio + aiohttp) enable high concurrency from a single process without threading overhead. The OpenAI API is the de facto standard: all major inference servers (vLLM, SGLang, TGI) expose compatible endpoints.


In [ ]:
import asyncio
import time
import json
import numpy as np
import matplotlib.pyplot as plt
from typing import AsyncIterator, Iterator

print('Client code patterns for inference')


## 1. Streaming with Server-Sent Events (SSE)

SSE streams token deltas as they're generated. The client receives partial responses and can display them immediately.


In [ ]:
def simulate_sse_stream(prompt: str, max_tokens: int = 100,
                       ttft_ms: float = 50, tpot_ms: float = 20) -> Iterator[str]:
    """
    Simulate an SSE stream from an OpenAI-compatible server.
    Yields JSON chunks like the real API.
    """
    tokens = [f'token_{i}' for i in range(max_tokens)]
    request_id = 'chatcmpl-abc123'
    model = 'llama-3-8b'

    # First token has TTFT latency
    time.sleep(ttft_ms / 1000)
    yield json.dumps({
        'id': request_id, 'model': model,
        'choices': [{'delta': {'role': 'assistant', 'content': tokens[0]}, 'finish_reason': None}]
    })

    for token in tokens[1:]:
        time.sleep(tpot_ms / 1000)
        yield json.dumps({
            'id': request_id, 'model': model,
            'choices': [{'delta': {'content': ' ' + token}, 'finish_reason': None}]
        })

    yield json.dumps({
        'id': request_id, 'model': model,
        'choices': [{'delta': {}, 'finish_reason': 'stop'}]
    })
    yield '[DONE]'

# Client that measures streaming metrics
def streaming_client(prompt: str, max_tokens: int = 20):
    collected = []
    first_token_time = None
    start = time.perf_counter()
    token_times = []

    for chunk in simulate_sse_stream(prompt, max_tokens):
        now = time.perf_counter()
        if chunk == '[DONE]':
            break
        data = json.loads(chunk)
        delta = data['choices'][0]['delta'].get('content', '')
        if delta:
            if first_token_time is None:
                first_token_time = now
            token_times.append(now)
            collected.append(delta)

    total_time = time.perf_counter() - start
    ttft = (first_token_time - start) * 1000 if first_token_time else 0
    tpot = np.mean(np.diff([t * 1000 for t in token_times])) if len(token_times) > 1 else 0

    print(f'Streaming client metrics:')
    print(f'  TTFT: {ttft:.1f} ms')
    print(f'  TPOT: {tpot:.1f} ms')
    print(f'  Total tokens: {len(collected)}')
    print(f'  Total time: {total_time*1000:.0f} ms')
    return collected

tokens = streaming_client('Tell me about inference engineering', max_tokens=10)


## 2. Async Batch Client

asyncio + aiohttp enables sending many concurrent requests from a single Python process — essential for load testing and batch processing.


In [ ]:
import asyncio

async def simulate_async_request(request_id: int, prompt_len: int,
                                  output_len: int, server_delay_ms: float = 50) -> dict:
    """Simulate an async HTTP request to an inference server."""
    start = time.perf_counter()
    await asyncio.sleep(server_delay_ms / 1000)  # TTFT
    await asyncio.sleep(output_len * 20 / 1000)  # decode
    elapsed = (time.perf_counter() - start) * 1000
    return {'id': request_id, 'latency_ms': elapsed, 'tokens': output_len}

async def batch_inference(requests: list, max_concurrency: int = 10):
    """Run batch inference with bounded concurrency."""
    semaphore = asyncio.Semaphore(max_concurrency)

    async def bounded_request(req):
        async with semaphore:
            return await simulate_async_request(**req)

    tasks = [bounded_request(req) for req in requests]
    results = await asyncio.gather(*tasks)
    return results

# Run benchmark
np.random.seed(42)
requests = [
    {'request_id': i, 'prompt_len': np.random.randint(50, 200),
     'output_len': np.random.randint(20, 100), 'server_delay_ms': 30}
    for i in range(50)
]

for concurrency in [1, 5, 10, 20]:
    start = time.perf_counter()
    results = asyncio.run(batch_inference(requests, max_concurrency=concurrency))
    total_time = (time.perf_counter() - start) * 1000
    latencies = [r['latency_ms'] for r in results]
    throughput = len(requests) / total_time * 1000
    print(f'Concurrency={concurrency:>3}: {total_time:>7.0f}ms total, '
          f'P50={np.percentile(latencies,50):>6.0f}ms, '
          f'{throughput:>5.1f} req/s')


## 3. Retry and Error Handling

Production clients need exponential backoff, timeout handling, and circuit breakers.


In [ ]:
import random

async def robust_inference_client(prompt: str, max_retries: int = 3,
                                   base_delay_s: float = 1.0,
                                   timeout_s: float = 30.0):
    """Production inference client with retry and timeout."""
    for attempt in range(max_retries):
        try:
            # Simulate random failures (429 rate limit, 503 overload)
            await asyncio.sleep(0.01)  # simulate network
            if random.random() < 0.2:  # 20% failure rate for demo
                error_code = random.choice([429, 503, 500])
                raise ConnectionError(f'HTTP {error_code}')
            return {'text': 'response', 'attempt': attempt + 1}

        except ConnectionError as e:
            if attempt == max_retries - 1:
                raise
            delay = base_delay_s * (2 ** attempt) + random.uniform(0, 1)
            print(f'  Attempt {attempt+1} failed ({e}), retrying in {delay:.1f}s...')
            await asyncio.sleep(min(delay, 10))

async def test_retry():
    random.seed(42)
    print('Retry logic demo (20% random failure rate):')
    for i in range(5):
        try:
            result = await robust_inference_client(f'prompt {i}')
            print(f'  Request {i}: SUCCESS on attempt {result["attempt"]}')
        except Exception as e:
            print(f'  Request {i}: FAILED after all retries: {e}')

asyncio.run(test_retry())


## 4. Protocol Comparison

REST, gRPC, and WebSocket have different tradeoffs for inference.


In [ ]:
protocols = {
    'REST/HTTP SSE': {
        'streaming': True,
        'latency': 'Low (HTTP/2 + SSE)',
        'throughput': 'Medium',
        'client_complexity': 'Low',
        'openai_compatible': True,
        'best_for': 'Chat interfaces, standard API clients',
    },
    'gRPC': {
        'streaming': True,
        'latency': 'Very low (HTTP/2 + protobuf)',
        'throughput': 'High',
        'client_complexity': 'Medium (code gen)',
        'openai_compatible': False,
        'best_for': 'Internal microservices, high-throughput batch',
    },
    'WebSocket': {
        'streaming': True,
        'latency': 'Low',
        'throughput': 'Medium',
        'client_complexity': 'Medium',
        'openai_compatible': False,
        'best_for': 'Interactive applications, bidirectional',
    },
    'REST batch': {
        'streaming': False,
        'latency': 'High (wait for full response)',
        'throughput': 'Medium',
        'client_complexity': 'Very low',
        'openai_compatible': True,
        'best_for': 'Offline processing, simple integration',
    },
}
for name, props in protocols.items():
    print(f'{name}:')
    for k, v in props.items():
        print(f'  {k:<25} {v}')
    print()


## Experiments: Try These

1. **Real SSE client**: Write a Python client using `httpx` with streaming that measures TTFT and TPOT from a live vLLM server.
2. **Load test with async**: Use the async batch client to send 1000 concurrent requests to your inference server. Find the concurrency level where P99 latency doubles.
3. **gRPC client**: Generate a gRPC client from the Triton Inference Server proto and compare latency to the REST endpoint.


## Key Takeaways

- SSE streaming converts inference from a blocking call to a token-by-token stream — reduces perceived latency by showing output immediately, not just at end of generation.
- Async clients (asyncio) enable high concurrency from a single Python process — essential for load testing and batch processing without threading overhead.
- Production clients need: exponential backoff for retries, request timeout, and circuit breakers — naive clients will amplify server overload with retries.
- OpenAI API compatibility (REST + SSE) is the de facto standard; any compliant client works with vLLM, SGLang, TGI, and NIMs.

## References
- *Inference Engineering* Ch 7.5 — Philip Kiely, Baseten Books 2026
- OpenAI API documentation
- httpx streaming documentation
- asyncio documentation
